# MV Validation: Label Split Leakage / Stability Check

This notebook checks whether the current MD label split may create leakage across train / validation / test due to the same employee-account pair or overlapping employee-account lookback windows appearing in multiple splits.

It addresses the email request:

1. Check if it is feasible to keep the same account-employee pair in one fold while maintaining label balance.
2. Check if it is feasible to keep overlapping account-employee windows in one fold while maintaining label balance.
3. If feasible, use the proposed label split for retraining and compare model performance against MD's original model performance.

This notebook is intended as MV validation work. It does not overwrite MD model artifacts.


## 1. Import libraries

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql import Window

## 2. Load feature table

If this notebook is appended to `lgbm_model.ipynb`, it will reuse `features_df` if it already exists.

If this is run as a standalone MV notebook, it reloads MD's saved in-time feature table.


In [ ]:
# Reuse features_df if running after lgbm_model.ipynb.
# Otherwise reload MD's saved in-time feature table.

if "features_df" not in globals():
    features_df = (
        spark.read.format("delta")
        .load(
            "abfss://ml-artifact-insider-us@dsapdafazprdadls1.dfs.core.windows.net/ins_us_nms/v1/output/insider_us_nms_features_table_v2"
        )
    )

display(features_df.limit(10))

## 3. Prepare split check dataframe

In [ ]:
split_check_df = (
    features_df
    .select(
        "login_id",
        "acct_nbr",
        "date",
        "lookback_window_start",
        "fraud_date",
        "label",
        "label_split",
        "cv_fold"
    )
    .withColumn("date", F.to_date("date"))
    .withColumn("lookback_window_start", F.to_date("lookback_window_start"))
    .withColumn("fraud_date", F.to_date("fraud_date"))
    .withColumn("window_start", F.col("lookback_window_start"))
    .withColumn("window_end", F.coalesce(F.col("fraud_date"), F.col("date")))
)

display(split_check_df.limit(10))

## 4. Current MD split label balance

This checks the current row count, positive count, negative count, positive rate, and number of unique employees/accounts/pairs in each original MD split.


In [ ]:
current_split_balance = (
    split_check_df
    .groupBy("label_split")
    .agg(
        F.count("*").alias("num_rows"),
        F.sum(F.col("label")).alias("num_positive"),
        (F.count("*") - F.sum(F.col("label"))).alias("num_negative"),
        F.mean(F.col("label")).alias("positive_rate"),
        F.countDistinct("login_id").alias("num_employees"),
        F.countDistinct("acct_nbr").alias("num_accounts"),
        F.countDistinct("login_id", "acct_nbr").alias("num_employee_account_pairs")
    )
    .orderBy("label_split")
)

display(current_split_balance)

## 5. Check if the same employee-account pair crosses multiple splits

This directly addresses:

> Check if it is feasible to ensure same account-employee in one-fold while maintaining label balance.

If `num_pairs_crossing_splits` is greater than 0, the same `login_id + acct_nbr` appears in more than one original MD split.


In [ ]:
pair_split_check = (
    split_check_df
    .groupBy("login_id", "acct_nbr")
    .agg(
        F.count("*").alias("num_rows"),
        F.countDistinct("label_split").alias("num_splits"),
        F.collect_set("label_split").alias("splits"),
        F.sum(F.col("label")).alias("num_positive"),
        F.mean(F.col("label")).alias("positive_rate"),
        F.min("date").alias("min_date"),
        F.max("date").alias("max_date")
    )
)

pair_cross_split = (
    pair_split_check
    .filter(F.col("num_splits") > 1)
    .orderBy(F.desc("num_splits"), F.desc("num_rows"))
)

display(pair_cross_split)

pair_cross_split_summary = (
    pair_split_check
    .agg(
        F.count("*").alias("num_employee_account_pairs"),
        F.sum(F.when(F.col("num_splits") > 1, 1).otherwise(0)).alias("num_pairs_crossing_splits"),
        F.mean(F.when(F.col("num_splits") > 1, 1).otherwise(0)).alias("pct_pairs_crossing_splits")
    )
)

display(pair_cross_split_summary)

## 6. Check if overlapping windows for the same employee-account pair cross splits

This addresses the stricter requirement:

> Check if it is feasible to ensure same account-employee overlapping in one-fold while maintaining label balance.

Two windows overlap when:

```text
window_start_a <= window_end_b
and
window_start_b <= window_end_a
```

If overlapping windows for the same `login_id + acct_nbr` are in different splits, this may create leakage or overly optimistic performance.


In [ ]:
window_base = (
    split_check_df
    .select(
        "login_id",
        "acct_nbr",
        "date",
        "window_start",
        "window_end",
        "label",
        "label_split"
    )
    .dropDuplicates()
    .withColumn("row_id", F.monotonically_increasing_id())
)

a = window_base.alias("a")
b = window_base.alias("b")

overlap_cross_split = (
    a.join(
        b,
        on=[
            F.col("a.login_id") == F.col("b.login_id"),
            F.col("a.acct_nbr") == F.col("b.acct_nbr"),
            F.col("a.row_id") < F.col("b.row_id"),
            F.col("a.label_split") != F.col("b.label_split"),
            F.col("a.window_start") <= F.col("b.window_end"),
            F.col("b.window_start") <= F.col("a.window_end")
        ],
        how="inner"
    )
    .select(
        F.col("a.login_id").alias("login_id"),
        F.col("a.acct_nbr").alias("acct_nbr"),

        F.col("a.date").alias("date_a"),
        F.col("a.window_start").alias("window_start_a"),
        F.col("a.window_end").alias("window_end_a"),
        F.col("a.label_split").alias("label_split_a"),
        F.col("a.label").alias("label_a"),

        F.col("b.date").alias("date_b"),
        F.col("b.window_start").alias("window_start_b"),
        F.col("b.window_end").alias("window_end_b"),
        F.col("b.label_split").alias("label_split_b"),
        F.col("b.label").alias("label_b")
    )
)

display(overlap_cross_split.limit(100))

overlap_cross_split_summary = (
    overlap_cross_split
    .agg(
        F.count("*").alias("num_overlapping_window_pairs_crossing_splits"),
        F.countDistinct("login_id", "acct_nbr").alias("num_employee_account_pairs_with_overlap_cross_split")
    )
)

display(overlap_cross_split_summary)

## 7. Proposed deterministic employee-account-level split

This creates a deterministic split based on a stable hash of `login_id + acct_nbr`.

Purpose:

- All rows for the same employee-account pair are assigned to one split.
- Split assignment is deterministic and reproducible.
- Label balance can be compared against the original MD split.


In [ ]:
proposed_split_df = (
    split_check_df
    .withColumn(
        "pair_hash_bucket",
        F.pmod(
            F.abs(F.xxhash64(F.col("login_id").cast("string"), F.col("acct_nbr").cast("string"))),
            F.lit(100)
        )
    )
    .withColumn(
        "mv_pair_level_split",
        F.when(F.col("pair_hash_bucket") < 70, F.lit("train"))
         .when(F.col("pair_hash_bucket") < 85, F.lit("val"))
         .otherwise(F.lit("test"))
    )
)

proposed_split_balance = (
    proposed_split_df
    .groupBy("mv_pair_level_split")
    .agg(
        F.count("*").alias("num_rows"),
        F.sum(F.col("label")).alias("num_positive"),
        (F.count("*") - F.sum(F.col("label"))).alias("num_negative"),
        F.mean(F.col("label")).alias("positive_rate"),
        F.countDistinct("login_id").alias("num_employees"),
        F.countDistinct("acct_nbr").alias("num_accounts"),
        F.countDistinct("login_id", "acct_nbr").alias("num_employee_account_pairs")
    )
    .orderBy("mv_pair_level_split")
)

display(proposed_split_balance)

## 8. Confirm no employee-account pair crosses the proposed split

Expected result:

```text
num_pairs_crossing_proposed_splits = 0
```


In [ ]:
proposed_pair_check = (
    proposed_split_df
    .groupBy("login_id", "acct_nbr")
    .agg(
        F.countDistinct("mv_pair_level_split").alias("num_proposed_splits"),
        F.collect_set("mv_pair_level_split").alias("proposed_splits")
    )
)

proposed_pair_leakage = (
    proposed_pair_check
    .filter(F.col("num_proposed_splits") > 1)
)

display(proposed_pair_leakage)

proposed_pair_leakage_summary = (
    proposed_pair_check
    .agg(
        F.count("*").alias("num_employee_account_pairs"),
        F.sum(F.when(F.col("num_proposed_splits") > 1, 1).otherwise(0)).alias("num_pairs_crossing_proposed_splits")
    )
)

display(proposed_pair_leakage_summary)

## 9. Compare original split balance vs proposed pair-level split balance

This helps determine whether the proposed pair-level split is feasible while maintaining label balance.


In [ ]:
display(current_split_balance)
display(proposed_split_balance)

## 10. Optional: create full feature dataframe with proposed MV split

This prepares train / validation / test datasets using the proposed pair-level split.

Do not overwrite MD's original `label_split`. This creates a new column: `mv_pair_level_split`.


In [ ]:
features_with_mv_split = (
    features_df
    .join(
        proposed_split_df
        .select(
            "login_id",
            "acct_nbr",
            "date",
            "mv_pair_level_split"
        )
        .dropDuplicates(),
        on=["login_id", "acct_nbr", "date"],
        how="left"
    )
)

mv_training = features_with_mv_split.filter(F.col("mv_pair_level_split") == "train")
mv_validation = features_with_mv_split.filter(F.col("mv_pair_level_split") == "val")
mv_testing = features_with_mv_split.filter(F.col("mv_pair_level_split") == "test")

print("MV proposed split counts:")
print("train:", mv_training.count())
print("val:", mv_validation.count())
print("test:", mv_testing.count())

display(
    features_with_mv_split
    .groupBy("mv_pair_level_split")
    .agg(
        F.count("*").alias("num_rows"),
        F.sum(F.col("label")).alias("num_positive"),
        F.mean(F.col("label")).alias("positive_rate"),
        F.countDistinct("login_id").alias("num_employees"),
        F.countDistinct("acct_nbr").alias("num_accounts"),
        F.countDistinct("login_id", "acct_nbr").alias("num_employee_account_pairs")
    )
    .orderBy("mv_pair_level_split")
)

## 11. Optional: save MV validation outputs

This saves MV-owned validation results as separate Delta tables.

These names are intentionally prefixed with `mv_` to avoid overwriting MD artifacts.


In [ ]:
# Optional save step. Uncomment if you want to persist MV validation outputs.

# current_split_balance.write.mode("overwrite").format("delta").saveAsTable(
#     "mv_insider_us_nms_current_split_balance"
# )

# pair_cross_split_summary.write.mode("overwrite").format("delta").saveAsTable(
#     "mv_insider_us_nms_pair_cross_split_summary"
# )

# overlap_cross_split_summary.write.mode("overwrite").format("delta").saveAsTable(
#     "mv_insider_us_nms_overlap_cross_split_summary"
# )

# proposed_split_balance.write.mode("overwrite").format("delta").saveAsTable(
#     "mv_insider_us_nms_proposed_pair_level_split_balance"
# )

# proposed_pair_leakage_summary.write.mode("overwrite").format("delta").saveAsTable(
#     "mv_insider_us_nms_proposed_pair_leakage_summary"
# )

## 12. Interpretation guide

Use this logic when writing the MV conclusion:

1. If `num_pairs_crossing_splits > 0`, the original MD split allows the same employee-account pair to appear in multiple splits.
2. If `num_overlapping_window_pairs_crossing_splits > 0`, overlapping lookback windows for the same employee-account pair cross splits.
3. If the proposed `mv_pair_level_split` has acceptable positive rates across train / val / test and `num_pairs_crossing_proposed_splits = 0`, then a pair-level deterministic split is feasible.
4. If feasible, the next step is to retrain the model using `mv_pair_level_split` and compare performance against MD's original model performance.
